# MNIST by GPT


In [98]:
# Simple MNIST in PyTorch (MLP)
# - Downloads MNIST
# - Builds a small neural network
# - Trains and evaluates accuracy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [99]:
# -----------------------------
# 1) Basic setup (device + seed)
# -----------------------------
torch.manual_seed(42)

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")


Using mps device


In [100]:
# -----------------------------------------
# 2) Data transforms (Tensor + normalization)
# -----------------------------------------
# MNIST pixels are 0..1 after ToTensor()
# Normalize with the standard MNIST mean/std


transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)

# assert MNIST pixels are 0..1 after ToTensor()
assert (
    datasets.MNIST(
        root="./data", train=True, download=True, transform=transforms.ToTensor()
    )[0][0].max()
    <= 1.0
), "MNIST pixels should be in the range [0, 1] after ToTensor()"


In [101]:
# -----------------------------
# 3) Download / load the dataset
# -----------------------------
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)


In [102]:
# -----------------------------
# 4) Define the model
# -----------------------------
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # MNIST images are 28x28 => 784 inputs
        self.net = nn.Sequential(
            nn.Flatten(),  # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(28 * 28, 256),  # hidden layer
            nn.ReLU(),
            nn.Linear(256, 10),  # 10 classes (digits 0-9)
        )

    def forward(self, x):
        return self.net(x)


model = SimpleMLP().to(device)


In [103]:
# ----------------------------------------
# 5) Loss function + optimizer
# ----------------------------------------
# CrossEntropyLoss expects raw logits (NO softmax in the model).
# It internally applies log-softmax + NLL loss.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [104]:
# ----------------------------------------
# 6) Training loop (one epoch)
# ----------------------------------------
def train_one_epoch(model, loader):
    model.train()  # enables training behaviors (e.g., dropout/bn if present)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        # Move batch to the right device
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass: compute logits
        logits = model(images)

        # Compute loss
        loss = criterion(logits, labels)

        # Backprop
        optimizer.zero_grad()  # clear old gradients
        loss.backward()  # compute new gradients
        optimizer.step()  # update weights

        # Track stats
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)  # predicted class = index of max logit
        total_correct += (preds == labels).sum().item()
        total_examples += images.size(0)

    avg_loss = total_loss / total_examples
    avg_acc = total_correct / total_examples
    return avg_loss, avg_acc


In [105]:
# ----------------------------------------
# 7) Evaluation loop
# ----------------------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()  # disables training behaviors
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_examples += images.size(0)

    return total_correct / total_examples


In [106]:
# ----------------------------------------
# 8) Run training
# ----------------------------------------
epochs = 5
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_acc = evaluate(model, test_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc * 100:.2f}% | "
        f"test acc: {test_acc * 100:.2f}%"
    )

# After a few epochs, you should typically see ~97-98% test accuracy with this MLP.

Epoch 01 | train loss: 0.2666 | train acc: 92.26% | test acc: 95.83%
Epoch 02 | train loss: 0.1105 | train acc: 96.74% | test acc: 97.06%
Epoch 03 | train loss: 0.0731 | train acc: 97.84% | test acc: 97.72%
Epoch 04 | train loss: 0.0538 | train acc: 98.32% | test acc: 97.72%
Epoch 05 | train loss: 0.0412 | train acc: 98.74% | test acc: 97.83%
